<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/04-backpropagation-automatic-differentiation.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Backpropagation and Automatic Differentiation** {#backpropagation-automatic-differentiation}

A neural network learns only when a final objective can be connected back to every parameter that influenced it. **Backpropagation** is the reverse traversal that applies the chain rule efficiently to that dependency graph. **Automatic differentiation (AD)** is the broader software technique that decomposes a program into differentiable primitives and mechanically composes their derivative rules. In ordinary deep-learning training, a framework uses reverse-mode AD to perform backpropagation from a scalar loss.

These terms are related but not identical. Backpropagation describes the gradient computation through a layered or graph-structured model; reverse-mode AD is the general algorithmic pattern; PyTorch autograd is one implementation with concrete contracts for graph construction, saved tensors, accumulation, and higher-order differentiation.

This chapter moves below `.backward()` and answers five practical questions:

1. What does a gradient say about a parameter, and what does it not say?
2. How are local derivative rules composed when a graph branches and rejoins?
3. Why does reverse mode avoid constructing enormous Jacobian matrices?
4. What information must be retained from the forward pass?
5. How can an implementation detect incorrect, vanishing, exploding, or disconnected gradients?

The optimization algorithm that uses these gradients is developed in Chapter 05. Here the emphasis is the correctness and numerical behavior of the gradient itself.

### **Learning as Credit Assignment** {#learning-credit-assignment}

Let a model with parameters $\theta \in \mathbb{R}^{P}$ produce predictions $f_\theta(X)$ and let a scalar loss measure their disagreement with targets:

$$
\mathcal{L}(\theta)=\ell(f_\theta(X),Y).
$$

The **credit-assignment problem** asks how much each parameter contributed to the current loss and in which local direction that loss would change. The answer is the gradient

$$
\nabla_\theta \mathcal{L}
=
\begin{bmatrix}
\partial \mathcal{L}/\partial \theta_1 & \cdots & \partial \mathcal{L}/\partial \theta_P
\end{bmatrix}^{\top}.
$$

For a small perturbation $\Delta\theta$, first-order Taylor expansion gives

$$
\mathcal{L}(\theta+\Delta\theta)
\approx
\mathcal{L}(\theta)
+
\nabla_\theta\mathcal{L}^{\top}\Delta\theta.
$$

The gradient is therefore the direction of steepest local increase under the Euclidean norm; $-\nabla_\theta\mathcal{L}$ is a local descent direction when the gradient is nonzero. A positive component does not mean that a parameter is "bad." It means that increasing that coordinate while holding the others fixed would locally increase the current loss. The interpretation changes under reparameterization, rescaling, a different batch, or a different objective.

Credit assignment is also not causal explanation. Parameter gradients describe local sensitivity of a specified computation. They do not by themselves establish that an input feature caused a real-world outcome, that a parameter is semantically interpretable, or that a finite update will have the effect predicted by a linear approximation.

<details>
<summary><strong>PyTorch: use gradients to test a local descent direction</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(31)

model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 1))
features = torch.randn(8, 3)
targets = torch.randn(8, 1)

loss_before = F.mse_loss(model(features), targets)
model.zero_grad()
loss_before.backward()

# Every trainable parameter should receive a finite gradient from this loss.
for name, parameter in model.named_parameters():
    assert parameter.grad is not None, f"missing gradient: {name}"
    assert torch.isfinite(parameter.grad).all(), f"non-finite gradient: {name}"

# Take a deliberately small step along the negative gradient.
step_size = 1e-3
with torch.no_grad():
    for parameter in model.parameters():
        parameter.add_(parameter.grad, alpha=-step_size)

loss_after = F.mse_loss(model(features), targets)
print("loss before:", float(loss_before.detach()))
print("loss after: ", float(loss_after.detach()))
assert loss_after < loss_before
~~~

</details>

The experiment checks a local statement, not a training strategy. A larger step can overshoot, and a gradient computed from one mini-batch may not reduce the population loss. Momentum, adaptive scaling, schedules, and stochasticity alter how credit becomes an update; those mechanisms belong to optimization rather than differentiation.

**Application.** Gradients train parameters, reveal disconnected modules, support sensitivity analysis, and provide building blocks for adversarial perturbations, influence approximations, meta-learning, and differentiable simulation.

**Comparison summary.** The loss assigns one scalar score to a complete prediction; the gradient distributes local sensitivity across parameters; an optimizer decides how to convert that sensitivity into a finite update. None of these alone supplies a causal explanation.

### **The Chain Rule on Computation Graphs** {#chain-rule-computation-graphs}

A complex model is differentiated by decomposing it into simple operations. If

$$
u=g(x), \qquad y=f(u),
$$

then the scalar chain rule states

$$
\frac{dy}{dx}
=
\frac{dy}{du}\frac{du}{dx}.
$$

The factor $du/dx$ is a **local derivative** known by operation $g$; $dy/du$ is the **upstream gradient** arriving from the rest of the graph. Their product is the gradient passed to $x$. No operation needs to understand the entire network. It only needs its local derivative rule and the upstream sensitivity.

A computation graph is normally a directed acyclic graph rather than a simple chain. Consider

$$
a=xy, \qquad b=x+y, \qquad f=ab.
$$

The variable $x$ influences $f$ through both $a$ and $b$. The multivariable chain rule adds the path contributions:

$$
\frac{\partial f}{\partial x}
=
\frac{\partial f}{\partial a}\frac{\partial a}{\partial x}
+
\frac{\partial f}{\partial b}\frac{\partial b}{\partial x}
=by+a.
$$

Similarly, $\partial f/\partial y=bx+a$. This is why gradient implementations use `+=` at forks. Overwriting a gradient keeps only one path and silently produces an incorrect result.

Graph structure also explains three recurring derivative patterns:

- an addition node copies its upstream gradient to both inputs;
- a multiplication node scales each input gradient by the other forward input;
- a broadcast operation sums gradients over every axis that was virtually expanded.

<details>
<summary><strong>PyTorch: compare path-wise chain-rule arithmetic with autograd</strong></summary>

~~~python
import torch

x_value, y_value = 2.0, -3.0

# Forward pass through a graph with two paths from each input to f.
a = x_value * y_value
b = x_value + y_value
f = a * b

# Reverse pass: f = a * b.
df_da = b
df_db = a

# Accumulate contributions through a = x*y and b = x+y.
manual_df_dx = df_da * y_value + df_db * 1.0
manual_df_dy = df_da * x_value + df_db * 1.0

x = torch.tensor(x_value, requires_grad=True)
y = torch.tensor(y_value, requires_grad=True)
output = (x * y) * (x + y)
output.backward()

assert output.item() == f
assert x.grad.item() == manual_df_dx
assert y.grad.item() == manual_df_dy
print("df/dx:", x.grad.item(), "df/dy:", y.grad.item())
~~~

</details>

The graph must be traversed in an order compatible with dependencies. Forward evaluation uses a topological order from inputs to outputs. Backpropagation uses the reverse order so that every node has received all downstream contributions before it propagates farther.

**Application.** The same rule handles residual branches, shared parameters, tied embeddings, recurrent unrolling, and multi-task heads. Whenever one tensor affects the loss by multiple routes, its final gradient is the sum of all route contributions.

**Comparison summary.** A chain multiplies local derivatives; a fork creates multiple paths whose contributions add; a computation graph records which composition rule applies. Backpropagation is dynamic programming over this graph, reusing intermediate sensitivities instead of enumerating every path separately.

### **Forward Pass and Local Derivatives** {#forward-pass-local-derivatives}

The **forward pass** evaluates the model in dependency order. During gradient-enabled execution it also records enough information to later apply local reverse rules. For a two-layer MLP,

$$
Z_1=XW_1^{\top}+b_1,
\qquad
H=\operatorname{ReLU}(Z_1),
\qquad
\widehat{Y}=HW_2^{\top}+b_2,
\qquad
\mathcal{L}=\ell(\widehat{Y},Y).
$$

![A forward computation graph orders variables and operations from input and parameters to the final objective.](assets/dl04-forward-computation-graph.svg){fig-align="center" width="78%" fig-alt="A forward computation graph from inputs and model parameters through matrix multiplication, activation, regularization, and the final objective."}

*Image source: [Dive into Deep Learning, Forward Propagation, Backward Propagation, and Computational Graphs](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

Different operations need different saved values:

| Forward operation | Output | Information commonly needed by backward |
|---|---|---|
| $y=x_1+x_2$ | sum | input shapes for broadcast reduction |
| $y=x_1x_2$ | product | both input values |
| $Y=XW^{\top}+b$ | affine output | $X$, $W$, and broadcast shape of $b$ |
| $y=\operatorname{ReLU}(x)$ | thresholded value | a mask such as $x>0$ |
| $y=\tanh(x)$ | bounded activation | $x$ or the already computed output $y$ |
| normalization | normalized tensor | statistics, inverse scale, and sometimes normalized activations |

The saved object is often called a **cache** or **context**. Saving everything minimizes recomputation but increases activation memory. Saving less can require recomputing part of the forward pass during backward. Activation checkpointing deliberately trades extra computation for lower memory by retaining selected boundaries and replaying internal operations later.

Forward values also determine which local rule is active. ReLU's derivative depends on whether its input was positive; max pooling must remember which index won; a data-dependent branch records only the operations that actually executed in an eager dynamic graph.

<details>
<summary><strong>PyTorch: build an explicit forward cache and account for its tensors</strong></summary>

~~~python
import torch
from torch.nn import functional as F


def mlp_forward_with_cache(x, weight_1, bias_1, weight_2, bias_2):
    """Return predictions plus exactly the values used by a manual backward pass."""
    pre_activation = x @ weight_1.T + bias_1
    hidden = F.relu(pre_activation)
    prediction = hidden @ weight_2.T + bias_2
    cache = {
        "x": x,
        "weight_1": weight_1,
        "pre_activation": pre_activation,
        "hidden": hidden,
        "weight_2": weight_2,
    }
    return prediction, cache


torch.manual_seed(37)
x = torch.randn(16, 8)
weight_1 = torch.randn(12, 8)
bias_1 = torch.randn(12)
weight_2 = torch.randn(4, 12)
bias_2 = torch.randn(4)

prediction, cache = mlp_forward_with_cache(x, weight_1, bias_1, weight_2, bias_2)
cache_bytes = sum(tensor.numel() * tensor.element_size() for tensor in cache.values())

assert prediction.shape == (16, 4)
assert cache["pre_activation"].shape == (16, 12)
print("cached tensors:", {name: tuple(value.shape) for name, value in cache.items()})
print("cache payload bytes:", cache_bytes)
~~~

</details>

This byte count includes tensors already owned elsewhere, so it is not a peak-memory profiler. It does expose what a backward formula depends on. Frameworks may save views, packed masks, fused intermediates, or recomputable metadata rather than the exact Python dictionary shown here.

**Application.** Understanding saved tensors explains why training consumes more memory than inference, why in-place modification can invalidate backward, and how checkpointing or fused custom kernels alter memory-compute tradeoffs.

**Comparison summary.** Forward evaluation computes values; gradient-enabled forward also builds derivative history and retains contexts. Caching reduces backward computation but costs memory, while recomputation saves memory at the cost of another partial forward pass.

### **Reverse-Mode Backpropagation** {#reverse-mode-backpropagation}

For every intermediate variable $v$, define its **adjoint** or reverse sensitivity

$$
\bar{v}=\frac{\partial \mathcal{L}}{\partial v}.
$$

Reverse mode begins at a scalar loss with seed $\bar{\mathcal{L}}=1$. It visits operations in reverse topological order. If a node computes $v=f(u_1,\ldots,u_k)$, its pullback performs

$$
\bar{u}_i
\mathrel{+}=
\left(\frac{\partial v}{\partial u_i}\right)^{\top}\bar{v}.
$$

The transpose notation matters for vector-valued intermediates. Conceptually, the node receives a sensitivity in its output space and pulls it back into each input space. The algorithm can be summarized as:

```text
1. Execute the forward graph and retain required contexts.
2. Topologically order nodes from leaves to the scalar loss.
3. Initialize every adjoint to zero and set loss.adjoint = 1.
4. Visit nodes in reverse topological order.
5. Apply each node's local pullback and accumulate into parent adjoints.
6. Read the adjoints of parameters and requested inputs.
```

For the MLP in the previous section and mean-squared error, let $G_{\widehat{Y}}=\partial\mathcal{L}/\partial\widehat{Y}$. The reverse equations are

$$
G_{W_2}=G_{\widehat{Y}}^{\top}H,
\quad
G_{b_2}=\sum_{b=1}^{B}G_{\widehat{Y},b},
\quad
G_H=G_{\widehat{Y}}W_2,
$$

$$
G_{Z_1}=G_H\odot\mathbf{1}[Z_1>0],
\quad
G_{W_1}=G_{Z_1}^{\top}X,
\quad
G_{b_1}=\sum_{b=1}^{B}G_{Z_1,b}.
$$

Every gradient has the same shape as the quantity it differentiates: $G_{W_1}$ has shape `[H, D_in]`, $G_{b_1}$ has shape `[H]`, and $G_X$ has shape `[B, D_in]`. Shape checking is one of the fastest ways to diagnose a manual backward derivation.

<details>
<summary><strong>PyTorch: derive a complete two-layer MLP backward pass by hand</strong></summary>

~~~python
import torch
from torch.nn import functional as F

torch.manual_seed(41)
B, D_IN, HIDDEN, D_OUT = 5, 3, 7, 2

x = torch.randn(B, D_IN)
target = torch.randn(B, D_OUT)
weight_1 = torch.randn(HIDDEN, D_IN)
bias_1 = torch.randn(HIDDEN)
weight_2 = torch.randn(D_OUT, HIDDEN)
bias_2 = torch.randn(D_OUT)

# Forward pass.
z_1 = x @ weight_1.T + bias_1
hidden = F.relu(z_1)
prediction = hidden @ weight_2.T + bias_2
loss = F.mse_loss(prediction, target, reduction="mean")

# Reverse pass. MSE averages over every prediction element.
grad_prediction = 2.0 * (prediction - target) / prediction.numel()
grad_weight_2 = grad_prediction.T @ hidden
grad_bias_2 = grad_prediction.sum(dim=0)
grad_hidden = grad_prediction @ weight_2
grad_z_1 = grad_hidden * (z_1 > 0)
grad_weight_1 = grad_z_1.T @ x
grad_bias_1 = grad_z_1.sum(dim=0)
grad_x = grad_z_1 @ weight_1

# Rebuild the same graph with tracked leaves and compare against autograd.
x_ref = x.clone().requires_grad_(True)
w1_ref = weight_1.clone().requires_grad_(True)
b1_ref = bias_1.clone().requires_grad_(True)
w2_ref = weight_2.clone().requires_grad_(True)
b2_ref = bias_2.clone().requires_grad_(True)
prediction_ref = F.relu(x_ref @ w1_ref.T + b1_ref) @ w2_ref.T + b2_ref
loss_ref = F.mse_loss(prediction_ref, target)
loss_ref.backward()

comparisons = [
    (grad_x, x_ref.grad),
    (grad_weight_1, w1_ref.grad),
    (grad_bias_1, b1_ref.grad),
    (grad_weight_2, w2_ref.grad),
    (grad_bias_2, b2_ref.grad),
]
assert all(torch.allclose(manual, automatic, atol=1e-6) for manual, automatic in comparisons)
print("manual backward matches autograd:", True)
~~~

</details>

After backward, many eager frameworks release saved graph state because it can be large. `retain_graph=True` preserves it for another reverse traversal, while `create_graph=True` records the derivative computation itself so that higher-order derivatives can be taken. These options solve different problems and both increase memory use.

**Application.** Reverse mode powers ordinary neural-network training, saliency gradients, gradient-based input optimization, and any scalar objective differentiated with respect to many parameters.

**Comparison summary.** Forward evaluation pushes values from inputs to loss; reverse mode pulls one loss sensitivity back to all ancestors. Accumulation handles shared paths, and cached local contexts prevent repeated symbolic expansion.

### **Jacobians, Vector-Jacobian Products, and Jacobian-Vector Products** {#jacobians-vjp-jvp}

For a vector function $f:\mathbb{R}^{n}\rightarrow\mathbb{R}^{m}$, the Jacobian is

$$
J_f(x)
=
\frac{\partial f}{\partial x}
\in\mathbb{R}^{m\times n},
\qquad
[J_f]_{ij}=\frac{\partial f_i}{\partial x_j}.
$$

Materializing $J_f$ is often wasteful. A layer mapping a million-dimensional activation to another million-dimensional activation would have a formal Jacobian with $10^{12}$ entries, even if its structure is sparse or its action can be computed cheaply. AD systems therefore expose Jacobian **products**.

A **Jacobian-vector product (JVP)** pushes an input tangent $u\in\mathbb{R}^{n}$ forward:

$$
J_f(x)u\in\mathbb{R}^{m}.
$$

It is the directional derivative of $f$ along $u$:

$$
J_f(x)u
=
\left.\frac{d}{d\epsilon}f(x+\epsilon u)\right|_{\epsilon=0}.
$$

A **vector-Jacobian product (VJP)** pulls an output cotangent $v\in\mathbb{R}^{m}$ backward:

$$
v^{\top}J_f(x)\in\mathbb{R}^{n}.
$$

When $m=1$ and $v=1$, the VJP is the familiar gradient of a scalar output. PyTorch's `.backward(gradient=...)` on a non-scalar output is also a VJP: the supplied tensor is the output cotangent seed.

<details>
<summary><strong>PyTorch: verify JVP and VJP without materializing a Jacobian</strong></summary>

~~~python
import torch
from torch.func import jacrev, jvp, vjp


def vector_function(x: torch.Tensor) -> torch.Tensor:
    return torch.stack(
        (
            x[0] * x[1] + torch.sin(x[2]),
            x[0].square() + torch.exp(x[1]),
        )
    )


x = torch.tensor([0.7, -0.4, 1.2], dtype=torch.float64)
input_tangent = torch.tensor([1.0, 2.0, -1.0], dtype=torch.float64)
output_cotangent = torch.tensor([0.5, -1.5], dtype=torch.float64)

output, jvp_result = jvp(vector_function, (x,), (input_tangent,))
output_again, pullback = vjp(vector_function, x)
vjp_result = pullback(output_cotangent)[0]

# The full Jacobian is formed only as a small reference check.
jacobian = jacrev(vector_function)(x)
assert jacobian.shape == (2, 3)
assert torch.allclose(jvp_result, jacobian @ input_tangent)
assert torch.allclose(vjp_result, output_cotangent @ jacobian)
assert torch.allclose(output, output_again)

print("JVP shape:", tuple(jvp_result.shape))
print("VJP shape:", tuple(vjp_result.shape))
~~~

</details>

The words *tangent* and *cotangent* describe how these objects transform mathematically. In code, both are represented by tensors with the corresponding primal shapes. The distinction becomes important when deriving transposes, batching derivative transforms, or composing custom rules.

**Application.** JVPs support directional sensitivities, implicit models, differential equations, and forward-over-reverse Hessian products. VJPs support scalar-loss training, adjoint methods, and gradients of selected weighted output combinations.

**Comparison summary.** A full Jacobian answers every input-output partial derivative at high storage cost. A JVP asks how outputs move along one input direction; a VJP asks how one weighted output sensitivity maps back to inputs. Modern AD normally computes the product required by the task.

### **Forward-Mode vs Reverse-Mode Differentiation** {#forward-mode-vs-reverse-mode}

Forward and reverse mode apply the same chain rule in opposite computational directions. Suppose evaluating $f:\mathbb{R}^{n}\to\mathbb{R}^{m}$ costs $C$.

| Mode | Propagated quantity | Approximate sweeps for a full Jacobian | Best-shaped problem |
|---|---|---:|---|
| Forward mode | one input tangent through JVPs | $n$ | few inputs, many outputs |
| Reverse mode | one output cotangent through VJPs | $m$ | many inputs, few outputs |

One forward-mode sweep computes the effect of one input direction on every output. One reverse-mode sweep computes the effect of every input on one weighted output direction. Neural-network training usually has millions of parameter inputs and one scalar loss, so reverse mode obtains all parameter gradients in roughly one forward and one backward-scale computation rather than one sweep per parameter.

The tradeoff is memory. Forward mode can propagate tangents alongside primals without storing the complete reverse tape. Reverse mode normally retains or recomputes intermediate contexts until backward reaches them. Exact cost depends on primitive structure, batching, compiler fusion, and checkpointing, but the input-output dimensional argument remains the right first decision.

Derivative modes can be composed. For scalar $g:\mathbb{R}^{n}\to\mathbb{R}$, its Hessian $H_g\in\mathbb{R}^{n\times n}$ is often too large to construct. A Hessian-vector product

$$
H_g(x)u

$$

can be computed by applying a JVP to the reverse-mode gradient function. This **forward-over-reverse** composition is useful in second-order optimization, curvature diagnostics, influence methods, and implicit differentiation.

<details>
<summary><strong>PyTorch: compute a Hessian-vector product by composing transforms</strong></summary>

~~~python
import torch
from torch.func import grad, hessian, jvp


def scalar_objective(x: torch.Tensor) -> torch.Tensor:
    interaction = 0.1 * x.sum().square()
    return (torch.sin(x) * x.square()).sum() + interaction


x = torch.tensor([0.2, -0.7, 1.1], dtype=torch.float64)
direction = torch.tensor([1.0, 0.5, -2.0], dtype=torch.float64)

gradient_function = grad(scalar_objective)  # reverse mode: R^n -> R^n
gradient_value, hessian_vector = jvp(gradient_function, (x,), (direction,))

# Construct the tiny full Hessian only to verify the product.
full_hessian = hessian(scalar_objective)(x)
assert gradient_value.shape == x.shape
assert torch.allclose(hessian_vector, full_hessian @ direction, atol=1e-10)
print("Hessian-vector product:", hessian_vector)
~~~

</details>

In ordinary autograd code, `create_graph=True` is needed when the returned gradient must itself remain differentiable. `retain_graph=True` merely preserves an existing tape for another traversal; it does not automatically make the derivative operations differentiable. Confusing these flags causes both correctness errors and unnecessary memory growth.

**Application.** Reverse mode is the default for scalar-loss model training. Forward mode is attractive for low-dimensional parameter sensitivity or wide-output simulators. Mixed modes avoid full Hessians and Jacobians in advanced optimization and scientific machine learning.

**Comparison summary.** Mode choice is governed primarily by input and output dimensionality. Reverse mode is not universally faster, and higher-order derivatives are best expressed as compositions of JVP and VJP transforms rather than explicit derivative matrices.

### **Gradients of Common Neural Network Layers** {#gradients-common-neural-network-layers}

Backpropagation becomes practical because frameworks implement local pullbacks for reusable primitives. Several patterns recur across architectures.

For an affine layer

$$
Y=XW^{\top}+b,
\quad
X\in\mathbb{R}^{B\times D_{in}},
\quad
W\in\mathbb{R}^{D_{out}\times D_{in}},
$$

and upstream gradient $G=\partial\mathcal{L}/\partial Y$,

$$
\frac{\partial\mathcal{L}}{\partial X}=GW,
\qquad
\frac{\partial\mathcal{L}}{\partial W}=G^{\top}X,
\qquad
\frac{\partial\mathcal{L}}{\partial b}=\sum_{i=1}^{B}G_i.
$$

The bias formula is a reduction because the forward pass broadcast one bias vector across the batch.

Elementwise activations multiply the upstream gradient elementwise:

$$
G_x=G_y\odot\phi'(x).
$$

For ReLU, $\phi'(x)=\mathbf{1}[x>0]$ under the common zero convention. For sigmoid output $s=\sigma(x)$, $\phi'(x)=s(1-s)$; for $t=\tanh(x)$, $\phi'(x)=1-t^2$. Reusing the forward output can avoid reevaluating the nonlinear function.

Softmax and cross-entropy are best differentiated as one fused expression. For logits $z$, one-hot target $q$, and probabilities $p=\operatorname{softmax}(z)$,

$$
\ell=-\sum_{k=1}^{K}q_k\log p_k,
\qquad
\frac{\partial\ell}{\partial z}=p-q.
$$

This compact result avoids explicitly storing the dense softmax Jacobian. With mean reduction over $B$ examples, divide by $B$.

Other layers follow structured accumulation rules. Embedding backward scatter-adds gradients into every referenced row, so repeated IDs accumulate. Convolution backward is another convolution/correlation-like operation with shared-weight accumulation across positions. Normalization backward couples values that shared the same mean and variance, which is why it cannot be treated as independent elementwise scaling.

<details>
<summary><strong>PyTorch: derive softmax-cross-entropy and affine gradients</strong></summary>

~~~python
import torch
from torch.nn import functional as F

torch.manual_seed(43)
B, D_IN, K = 6, 4, 3
x = torch.randn(B, D_IN, dtype=torch.float64)
weight = torch.randn(K, D_IN, dtype=torch.float64)
bias = torch.randn(K, dtype=torch.float64)
targets = torch.tensor([0, 2, 1, 1, 0, 2])

logits = x @ weight.T + bias
probabilities = logits.softmax(dim=1)

# Fused cross-entropy derivative: (probability - one_hot_target) / B.
grad_logits = probabilities.clone()
grad_logits[torch.arange(B), targets] -= 1.0
grad_logits /= B

manual_grad_weight = grad_logits.T @ x
manual_grad_bias = grad_logits.sum(dim=0)
manual_grad_x = grad_logits @ weight

x_ref = x.clone().requires_grad_(True)
weight_ref = weight.clone().requires_grad_(True)
bias_ref = bias.clone().requires_grad_(True)
loss = F.cross_entropy(x_ref @ weight_ref.T + bias_ref, targets)
loss.backward()

assert torch.allclose(manual_grad_weight, weight_ref.grad, atol=1e-10)
assert torch.allclose(manual_grad_bias, bias_ref.grad, atol=1e-10)
assert torch.allclose(manual_grad_x, x_ref.grad, atol=1e-10)
print("fused derivative matches autograd:", True)
~~~

</details>

Custom kernels must return one gradient per differentiable input, reduce broadcast dimensions correctly, and preserve dtype/device contracts. A locally plausible derivative can still be globally wrong if its shape or accumulation semantics are incorrect.

**Application.** Knowing these rules helps derive custom layers, audit fused kernels, estimate backward cost, and recognize why certain tensors must be saved. It also makes autograd output less opaque during debugging.

**Comparison summary.** Elementwise layers mask or rescale upstream gradients; affine layers use matrix contractions; broadcast parameters reduce expanded axes; shared parameters accumulate across every use; fused losses exploit algebra to avoid full Jacobians.

### **A Micro-Autograd Engine from Scratch** {#micro-autograd-engine}

A minimal reverse-mode engine needs only four ideas:

1. a value stores its forward number and accumulated gradient;
2. every operation creates an output node linked to its parents;
3. the output node owns a local `_backward` function that updates parent gradients;
4. `.backward()` builds a topological order, seeds the final gradient with 1, and executes local rules in reverse.

![Micrograd visualizes a scalar neuron as a dynamic graph whose nodes carry both forward data and reverse gradients.](assets/micrograd-neuron-graph.svg){fig-align="center" width="100%" fig-alt="A wide Micrograd computation graph for a two-input neuron, showing scalar operations, forward values, and gradients."}

*Image source: Andrej Karpathy, [Micrograd](https://github.com/karpathy/micrograd), `gout.svg`, MIT License.*

The engine below is scalar-valued so that every local derivative is visible. Tensor frameworks apply the same architecture to vectorized kernels whose pullbacks perform matrix, reduction, scatter, or convolution operations.

<details>
<summary><strong>Python: implement scalar reverse-mode automatic differentiation</strong></summary>

~~~python
import math
import torch


class Value:
    """A scalar value with a dynamically constructed reverse-mode graph."""

    def __init__(self, data, children=(), operation="", label=""):
        self.data = float(data)
        self.grad = 0.0
        self.parents = tuple(children)
        self.operation = operation
        self.label = label
        self._backward = lambda: None

    @staticmethod
    def _coerce(other):
        return other if isinstance(other, Value) else Value(other)

    def __add__(self, other):
        other = self._coerce(other)
        output = Value(self.data + other.data, (self, other), "+")

        def backward():
            self.grad += output.grad
            other.grad += output.grad

        output._backward = backward
        return output

    def __mul__(self, other):
        other = self._coerce(other)
        output = Value(self.data * other.data, (self, other), "*")

        def backward():
            self.grad += other.data * output.grad
            other.grad += self.data * output.grad

        output._backward = backward
        return output

    def __pow__(self, exponent):
        output = Value(self.data**exponent, (self,), f"**{exponent}")

        def backward():
            self.grad += exponent * self.data ** (exponent - 1) * output.grad

        output._backward = backward
        return output

    def tanh(self):
        value = math.tanh(self.data)
        output = Value(value, (self,), "tanh")

        def backward():
            self.grad += (1.0 - value**2) * output.grad

        output._backward = backward
        return output

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-self._coerce(other))

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def backward(self):
        topological_order = []
        visited = set()

        def build(node):
            if node not in visited:
                visited.add(node)
                for parent in node.parents:
                    build(parent)
                topological_order.append(node)

        build(self)
        self.grad = 1.0
        for node in reversed(topological_order):
            node._backward()


# A two-input tanh neuron: y = tanh(x1*w1 + x2*w2 + b).
x_1 = Value(1.5, label="x1")
x_2 = Value(-2.0, label="x2")
w_1 = Value(0.7, label="w1")
w_2 = Value(-0.3, label="w2")
bias = Value(0.2, label="b")
output = (x_1 * w_1 + x_2 * w_2 + bias).tanh()
output.backward()

# Compare all leaf gradients against the equivalent PyTorch graph.
torch_leaves = [torch.tensor(v.data, requires_grad=True) for v in (x_1, x_2, w_1, w_2, bias)]
tx1, tx2, tw1, tw2, tbias = torch_leaves
torch_output = torch.tanh(tx1 * tw1 + tx2 * tw2 + tbias)
torch_output.backward()

micrograd_leaves = (x_1, x_2, w_1, w_2, bias)
assert all(abs(value.grad - tensor.grad.item()) < 1e-6 for value, tensor in zip(micrograd_leaves, torch_leaves))
print("output:", round(output.data, 6))
print("leaf gradients:", [round(value.grad, 6) for value in micrograd_leaves])
~~~

</details>

Two implementation details are easy to miss. First, gradients use `+=` because a node may be reused. Second, topological ordering ensures that all contributions to a node's output gradient are available before its pullback runs. A recursive expression expansion without memoization would repeat work exponentially on a graph with shared subexpressions.

Production engines add tensor shapes, devices, dtype promotion, views, mutation/version checks, parallel scheduling, custom kernels, graph pruning, saved-tensor hooks, compiled execution, and higher-order differentiation. The small engine is not a replacement; it isolates the invariant beneath those systems.

**Application.** Building a scalar engine turns autograd from an opaque service into a concrete graph algorithm. It is also a useful reference when implementing a custom `autograd.Function` or debugging missing accumulation.

**Comparison summary.** Manual differentiation writes one backward program for one model; a micro-engine attaches reusable pullbacks to primitive operations; a production tensor engine applies the same graph logic with vectorized kernels and systems-level state management.

### **Gradient Checking** {#gradient-checking}

Gradient checking compares an analytical or automatic gradient against a numerical approximation. For scalar $f$ and coordinate $i$, the central difference is

$$
\frac{\partial f}{\partial x_i}
\approx
\frac{f(x+\epsilon e_i)-f(x-\epsilon e_i)}{2\epsilon}.
$$

Central differences have truncation error of order $O(\epsilon^2)$, but making $\epsilon$ arbitrarily small increases floating-point cancellation. Good checks usually use `float64`, a moderate epsilon such as $10^{-6}$, deterministic computation, and relative as well as absolute error:

$$
\operatorname{relative\ error}
=
\frac{|g_{analytic}-g_{numeric}|}
{\max(1,|g_{analytic}|,|g_{numeric}|)}.
$$

For high-dimensional inputs, checking every coordinate is expensive. A directional check compares

$$
\nabla f(x)^{\top}u
\quad\text{with}\quad
\frac{f(x+\epsilon u)-f(x-\epsilon u)}{2\epsilon},
$$

which tests all coordinates in a random direction using two function evaluations. It can miss a specially aligned error, so several directions or sampled coordinates are safer.

Checks must avoid nondifferentiable boundaries. ReLU at exactly zero, max ties, discrete indexing decisions, clipping thresholds, and stochastic sampling can produce valid but convention-dependent discrepancies. Dropout and random augmentation should be disabled or controlled.

<details>
<summary><strong>PyTorch: validate a custom backward rule with gradcheck and a direction</strong></summary>

~~~python
import torch
from torch.autograd import Function, gradcheck


class CubicWithBias(Function):
    @staticmethod
    def forward(ctx, x, bias):
        ctx.save_for_backward(x)
        return x**3 + bias

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad_x = grad_output * 3.0 * x.square()
        grad_bias = grad_output.sum()  # bias was broadcast over x
        return grad_x, grad_bias


def objective(x, bias):
    return torch.sin(CubicWithBias.apply(x, bias)).sum()


x = torch.tensor([0.2, -0.8, 1.3], dtype=torch.float64, requires_grad=True)
bias = torch.tensor(0.15, dtype=torch.float64, requires_grad=True)

# gradcheck perturbs each double-precision input and compares Jacobian entries.
assert gradcheck(CubicWithBias.apply, (x, bias), eps=1e-6, atol=1e-5, rtol=1e-4)

# A separate directional check validates the complete scalar objective.
direction = torch.tensor([1.0, -0.5, 2.0], dtype=torch.float64)
analytical_gradient = torch.autograd.grad(objective(x, bias), x)[0]
analytical_directional = analytical_gradient @ direction

epsilon = 1e-6
with torch.no_grad():
    numerical_directional = (
        objective(x + epsilon * direction, bias)
        - objective(x - epsilon * direction, bias)
    ) / (2.0 * epsilon)

assert torch.allclose(analytical_directional, numerical_directional, atol=1e-7, rtol=1e-6)
print("directional derivative error:", float((analytical_directional - numerical_directional).abs()))
~~~

</details>

A passing numerical check raises confidence in a local rule but does not prove the model is semantically correct. The loss may use the wrong label, a tensor may be aligned with the wrong axis, or both analytical and numerical code may share the same upstream mistake.

**Application.** Gradient checking is most valuable for new custom operations, unusual broadcasting, implicit solvers, differentiable rendering, and hand-derived losses. It should run on tiny deterministic inputs in tests, not inside routine training.

**Comparison summary.** Autograd computes derivatives efficiently from local rules; finite differences estimate them from repeated evaluations; gradient checking compares the two. Numerical agreement validates local calculus, while separate tests must validate shapes and model meaning.

### **Vanishing and Exploding Gradients** {#vanishing-exploding-gradients}

In a deep composition

$$
h^{(l)}=f_l(h^{(l-1)}),
$$

the sensitivity from layer $L$ to an earlier layer $l$ contains a product of Jacobians:

$$
\frac{\partial h^{(L)}}{\partial h^{(l)}}
=
J_LJ_{L-1}\cdots J_{l+1}.
$$

If the relevant singular values are repeatedly below 1, gradient norms tend to shrink exponentially; if repeatedly above 1, they can grow exponentially. Orientation matters as well as magnitude, so eigenvalue or scalar intuition is only an approximation, but the product mechanism is fundamental.

![The sigmoid function has substantial derivative only near zero and nearly vanishing derivative in its saturated tails.](assets/dl04-sigmoid-gradient.svg){fig-align="center" width="68%" fig-alt="A plot comparing sigmoid values with sigmoid derivatives, showing derivatives approaching zero for large positive and negative inputs."}

*Image source: [Dive into Deep Learning, Numerical Stability and Initialization](https://d2l.ai/chapter_multilayer-perceptrons/numerical-stability-and-init.html), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

Sigmoid illustrates the activation contribution: $\sigma'(x)=\sigma(x)(1-\sigma(x))\leq 1/4$, and the derivative approaches zero in either saturated tail. Repeating such factors can block credit from reaching early layers. Weight matrices can amplify or contract the signal further. Recurrent networks reuse a transition across many time steps, so the same instability can be repeated hundreds of times.

Typical symptoms differ:

| Problem | Observable symptoms | Consequence |
|---|---|---|
| Vanishing gradient | early-layer norms near zero; features barely change | slow or absent long-range learning |
| Exploding gradient | sudden large norms, loss spikes, Inf/NaN values | destructive updates and numerical overflow |
| Dead/disconnected path | gradient is exactly zero or `None` | parameter receives no credit from the objective |
| Poor conditioning | very different norms across directions/layers | unstable progress and sensitivity to step size |

Mitigations address different causes. Xavier or He initialization controls initial variance; non-saturating activations preserve useful local slopes; residual connections add identity paths; normalization regulates activation scale; LSTM/GRU gates create controlled memory routes; gradient clipping limits an exploding update; and shorter effective paths reduce repeated products. Clipping treats the magnitude symptom but does not repair a fundamentally vanishing or disconnected route.

<details>
<summary><strong>PyTorch: observe gradient products in plain and residual scalar chains</strong></summary>

~~~python
import torch


def plain_chain_gradient(scale: float, depth: int) -> float:
    x = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
    hidden = x
    for _ in range(depth):
        hidden = scale * hidden
    return torch.autograd.grad(hidden, x)[0].item()


def residual_chain_gradient(scale: float, depth: int) -> float:
    x = torch.tensor(0.5, dtype=torch.float64, requires_grad=True)
    hidden = x
    for _ in range(depth):
        # Each local derivative contains an identity contribution of 1.
        hidden = hidden + scale * torch.tanh(hidden)
    return torch.autograd.grad(hidden, x)[0].item()


depth = 40
vanishing = plain_chain_gradient(scale=0.5, depth=depth)
exploding = plain_chain_gradient(scale=1.2, depth=depth)
residual = residual_chain_gradient(scale=0.01, depth=depth)

assert abs(vanishing - 0.5**depth) < 1e-15
assert abs(exploding - 1.2**depth) < 1e-8
assert 1.0 < residual < 2.0

print("plain scale 0.5:", f"{vanishing:.3e}")
print("plain scale 1.2:", f"{exploding:.3e}")
print("small residual updates:", f"{residual:.3f}")
~~~

</details>

Real networks require measurement rather than inference from loss alone. Track per-layer gradient norms, parameter-to-update ratios, activation distributions, and non-finite values. Compare several steps and seeds: a single zero can be legitimate after ReLU or masking, whereas persistent layer-wide zeros indicate a structural problem.

**Application.** Gradient-flow diagnosis is essential in very deep networks, recurrent and state-space models, long-context systems, mixed-precision training, and custom architectures where no established initialization recipe exists.

**Comparison summary.** Vanishing and exploding gradients are consequences of repeated Jacobian products; disconnected gradients arise from graph structure; non-finite gradients can also originate in unstable arithmetic. Initialization, architecture, normalization, clipping, and numerical precision solve different parts of the problem.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Backpropagation is best understood as a graph algorithm that transports local sensitivity, not as one enormous symbolic derivative. Forward execution creates values and derivative contexts; reverse execution consumes one output sensitivity and accumulates contributions into every ancestor.

| Concept | Primary object | Direction | Main resource cost | Frequent mistake |
|---|---|---|---|---|
| Chain rule | local derivative and upstream gradient | follows dependency composition | arithmetic per graph edge | forgetting to add branch contributions |
| Forward pass | primal values and saved contexts | inputs to outputs | activation computation and storage | discarding or mutating required values |
| Reverse mode | adjoints / VJPs | outputs to inputs | saved activations plus pullbacks | wrong seed, shape, or accumulation |
| Forward mode | tangents / JVPs | inputs to outputs | one tangent propagation per direction | using it for millions of independent inputs |
| Full Jacobian | all output-input partials | both dimensions exposed | potentially $O(mn)$ storage | constructing it when only a product is needed |
| Micro-autograd | dynamic DAG plus local closures | reverse topological | one scalar node per primitive | missing topological order or `+=` |
| Gradient check | finite-difference estimate | repeated forward evaluations | expensive but simple | testing stochastic or nondifferentiable points |
| Gradient-flow diagnosis | norms and finite-value checks | across layers/time | monitoring overhead | treating clipping as a universal cure |

The main conclusions are:

1. A gradient is a local sensitivity of a specified scalar objective, not a causal explanation or a guaranteed finite-step improvement.
2. Local derivatives multiply along a path and add across branches that share an ancestor.
3. Reverse mode seeds a scalar loss with 1 and propagates VJPs in reverse topological order.
4. Forward caches reduce recomputation but make training memory larger than inference memory.
5. JVPs and VJPs compute the action of a Jacobian without materializing the full matrix.
6. Reverse mode fits many-parameter, scalar-loss training; forward mode fits low-input, wide-output sensitivity; mixed modes support higher-order products.
7. Layer pullbacks must reproduce forward broadcasting, parameter sharing, and reductions exactly.
8. A small autograd engine requires graph construction, local closures, topological ordering, a seed, and gradient accumulation.
9. Finite differences are a diagnostic reference whose reliability depends on precision, epsilon, determinism, and smooth evaluation points.
10. Vanishing, exploding, disconnected, and non-finite gradients have different causes and require different interventions.

Chapter 05 turns correct gradients into a complete learning process: it defines loss functions, initialization and update rules, optimizer state, learning-rate schedules, mixed precision, and the dynamics that determine whether repeated updates converge.